# Image exemplar bank for RAG-grounded flood classification

Builds a season-masked, class-balanced, diversity-sampled image exemplar bank for the
E-Noe / Mineirinho Creek water-level classifier (see the `llm_flood_detection` repo).

**Selection rule (per season, independently):**
- Take **all** `medium` / `high` / `flood` images -- no dedup, no subsampling.
- Take **all** `low` images' embeddings, then keep only a diverse subset via greedy
  farthest-point sampling, sized to match: `n_low = n_medium + n_high + n_flood`
  (computed within that season, so seasons balance independently of each other).

Balancing *per season* (not globally) matters because retrieval later masks out
whichever season is the current leave-one-season-out test fold -- doing the balance
per season means every fold's masked bank is a sum of already-balanced chunks,
regardless of which season gets excluded.

**Output:** a persistent Chroma collection (`flood_image_examples`) with precomputed
DINOv2 embeddings + metadata (`season`, `place`, `label`, `datetime`, `is_night`, `path`,
`role`), zipped for download and re-import into the repo's `knowledge_base/`.

Run this on Kaggle with a GPU accelerator enabled (Settings -> Accelerator -> GPU).


## 1. Setup

In [ ]:
!pip install -q chromadb


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


## 2. Config

Adjust `IMAGES_PREFIX` if the verification cell below fails to find a sample image --
this mirrors `KAGGLE_PREFIX` in the repo's `data/enoe_images.py`, but that constant was
written against a different Kaggle mount path than the one given for this run, so it's
re-verified here rather than assumed.


In [ ]:
DATASET_ROOT = Path("/kaggle/input/datasets/caetanoranieri/river-images-at-sao-carlos")
CSV_PATH = DATASET_ROOT / "flood_images_annot_2.csv"
IMAGES_PREFIX = "enoe/enoe2"  # CSV `path` values are relative to this, under DATASET_ROOT

OUTPUT_DIR = Path("/kaggle/working")
INDEX_DIR = OUTPUT_DIR / "image_bank_index"
BACKUP_DIR = OUTPUT_DIR / "image_bank_backup"

LABELS = ["low", "medium", "high", "flood"]
RARE_LABELS = ["medium", "high", "flood"]

# Leave-one-season-out seasons, matching the source paper's split table exactly
# (kept in sync manually with data/enoe_images.py:SEASONS in the repo).
SEASONS = ["2018-2019", "2019-2020", "2020-2021", "2021-2022"]

FPS_SEED = 0
EMBED_BATCH_SIZE = 128
CHROMA_ADD_BATCH_SIZE = 500


## 3. Load annotations, resolve image paths, sanity-check counts

In [ ]:
def season_for(dt: pd.Timestamp) -> str:
    """Nov-Feb rainy season -- a November date belongs to the season starting that
    year; a Jan/Feb date belongs to the season that started the previous November.
    Must match data/enoe_images.py:season_for exactly."""
    year = dt.year if dt.month >= 11 else dt.year - 1
    return f"{year}-{year + 1}"


df = pd.read_csv(CSV_PATH, index_col=0, parse_dates=["datetime"])
df["season"] = df["datetime"].apply(season_for)
df["hour"] = df["datetime"].dt.hour
df["is_night"] = (df["hour"] < 6) | (df["hour"] >= 18)
df["image_path"] = df["path"].apply(lambda p: str(DATASET_ROOT / IMAGES_PREFIX / p))

print(f"Loaded {len(df)} rows")
assert df["season"].isin(SEASONS).all(), "unexpected season labels -- check season_for()"


In [ ]:
# Verify image resolution against a real file before spending GPU time on the full pass.
sample_row = df.iloc[0]
sample_path = Path(sample_row["image_path"])
if not sample_path.exists():
    raise FileNotFoundError(
        f"""Sample image not found at {sample_path}.
IMAGES_PREFIX is probably wrong for this dataset mount -- list DATASET_ROOT to find the
real image directory and update IMAGES_PREFIX above, e.g.:
    list(DATASET_ROOT.iterdir())
    list((DATASET_ROOT / "some_dir").rglob("*.jpg"))[:5]"""
    )
print("OK, sample image resolves:", sample_path)


In [ ]:
print("Overall label counts:")
print(df["level"].value_counts())
print()
print("Per-season row counts:")
print(df["season"].value_counts().reindex(SEASONS))
print()
print("Per-season rare-class counts:")
print(df[df["level"].isin(RARE_LABELS)].groupby(["season", "level"]).size().unstack(fill_value=0).reindex(SEASONS))


Expected (from the repo's `data/README.md`, full labeled set): `low=67803, medium=535,
high=186, flood=75`; season row counts `2018-2019=8175, 2019-2020=7755,
2020-2021=24647, 2021-2022=28022`. If these don't match, stop and check the CSV/path
resolution before continuing -- everything downstream assumes this is right.


## 4. DINOv2 embedder

In [ ]:
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
dino.eval().to(DEVICE)

preprocess = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


@torch.no_grad()
def embed_paths(paths: list[str], batch_size: int = EMBED_BATCH_SIZE) -> np.ndarray:
    """Batch-embed a list of image paths with DINOv2, returns (N, 768) float32."""
    all_embeddings = []
    for i in tqdm(range(0, len(paths), batch_size), desc="embedding", leave=False):
        batch_paths = paths[i : i + batch_size]
        tensors = []
        for p in batch_paths:
            with Image.open(p) as img:
                tensors.append(preprocess(img.convert("RGB")))
        batch = torch.stack(tensors).to(DEVICE)
        out = dino(batch)  # (B, 768) pooled CLS-token embedding for vitb14
        all_embeddings.append(out.cpu().numpy())
    return np.concatenate(all_embeddings, axis=0).astype(np.float32)


## 5. Greedy farthest-point (k-center) diversity sampling

In [ ]:
def farthest_point_sampling(embeddings: np.ndarray, k: int, seed: int = FPS_SEED) -> np.ndarray:
    """Greedy k-center: pick a seed, then repeatedly add the point maximizing minimum
    distance to everything already selected. Returns the k selected row indices into
    `embeddings`. If k >= n, returns all indices (no sampling needed)."""
    n = embeddings.shape[0]
    if k >= n:
        return np.arange(n)

    emb = torch.from_numpy(embeddings).to(DEVICE)
    rng = np.random.default_rng(seed)
    selected = [int(rng.integers(n))]
    min_dists = torch.cdist(emb, emb[selected[-1] : selected[-1] + 1]).squeeze(1)

    for _ in range(k - 1):
        next_idx = int(torch.argmax(min_dists).item())
        selected.append(next_idx)
        new_dists = torch.cdist(emb, emb[next_idx : next_idx + 1]).squeeze(1)
        min_dists = torch.minimum(min_dists, new_dists)

    return np.array(selected)


## 6. Build the bank, season by season

For each season: embed all rare-class images (kept in full, no dedup), embed all `low`
images, then keep only a farthest-point-sampled subset of `low` sized to
`n_medium + n_high + n_flood` for that season.


In [ ]:
records = []  # each: dict of metadata, embedding stored separately in `bank_embeddings`
bank_embeddings = []

for season in SEASONS:
    season_df = df[df["season"] == season]
    rare_df = season_df[season_df["level"].isin(RARE_LABELS)]
    low_df = season_df[season_df["level"] == "low"]
    n_rare = len(rare_df)

    print(f"\n=== season {season}: rare={n_rare} ({rare_df['level'].value_counts().to_dict()}), "
          f"low_total={len(low_df)}, low_target={n_rare} ===")

    if n_rare == 0:
        print(f"  skipping low sampling for {season} -- no rare-class images this season")
        continue

    rare_embeddings = embed_paths(rare_df["image_path"].tolist())
    for (_, row), emb in zip(rare_df.iterrows(), rare_embeddings):
        records.append({
            "season": season, "place": row["place"], "label": row["level"],
            "datetime": row["datetime"].isoformat(), "is_night": bool(row["is_night"]),
            "path": row["path"], "role": "rare",
        })
        bank_embeddings.append(emb)

    low_embeddings = embed_paths(low_df["image_path"].tolist())
    selected_idx = farthest_point_sampling(low_embeddings, k=n_rare)
    print(f"  selected {len(selected_idx)}/{len(low_df)} diverse low examples")

    low_rows = low_df.reset_index(drop=True)
    for idx in selected_idx:
        row = low_rows.iloc[idx]
        records.append({
            "season": season, "place": row["place"], "label": row["level"],
            "datetime": row["datetime"].isoformat(), "is_night": bool(row["is_night"]),
            "path": row["path"], "role": "low_diverse",
        })
        bank_embeddings.append(low_embeddings[idx])

bank_embeddings = np.stack(bank_embeddings, axis=0)
print(f"\nTotal bank size: {len(records)} ({bank_embeddings.shape})")


## 7. Persist: Chroma collection + backup parquet/npy, then zip for download

In [ ]:
import chromadb

INDEX_DIR.mkdir(parents=True, exist_ok=True)
client = chromadb.PersistentClient(path=str(INDEX_DIR))
try:
    client.delete_collection("flood_image_examples")
except Exception:
    pass
collection = client.create_collection("flood_image_examples")

ids = [f"{r['label']}_{r['season']}_{r['place']}_{i}" for i, r in enumerate(records)]

for i in tqdm(range(0, len(records), CHROMA_ADD_BATCH_SIZE), desc="writing to chroma"):
    batch_ids = ids[i : i + CHROMA_ADD_BATCH_SIZE]
    batch_embeddings = bank_embeddings[i : i + CHROMA_ADD_BATCH_SIZE].tolist()
    batch_metadatas = records[i : i + CHROMA_ADD_BATCH_SIZE]
    collection.add(ids=batch_ids, embeddings=batch_embeddings, metadatas=batch_metadatas)

print(f"Wrote {collection.count()} vectors to {INDEX_DIR}")


In [ ]:
# Independent backup (metadata + raw embeddings) in case the Chroma format needs
# re-importing differently later -- cheap insurance, not required to use the bank.
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(records).to_parquet(BACKUP_DIR / "metadata.parquet", index=False)
np.save(BACKUP_DIR / "embeddings.npy", bank_embeddings)
print(f"Backup written to {BACKUP_DIR}")


In [ ]:
# Zip just the two output dirs (not all of /kaggle/working) to keep the archive small.
zip_path = OUTPUT_DIR / "image_bank_output"
import zipfile
with zipfile.ZipFile(f"{zip_path}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for d in (INDEX_DIR, BACKUP_DIR):
        for p in d.rglob("*"):
            if p.is_file():
                zf.write(p, p.relative_to(OUTPUT_DIR))

print(f"Download {zip_path}.zip from the notebook's Output panel.")


## Next steps (back in the repo, not on Kaggle)

1. Download `image_bank_output.zip` from this notebook's Output panel.
2. Unzip into the repo, e.g. `knowledge_base/image_index/` (so
   `image_bank_index/` sits alongside the existing text `knowledge_base/index/`).
3. At retrieval time, query this collection with the target image's own DINOv2
   embedding and mask out the current leave-one-season-out test season:
   `collection.query(query_embeddings=[...], where={"season": {"$ne": test_season}})`.
